In [11]:
import re
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split

# Data Preprocessing

In [12]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("amirzenoozi/persian-news-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/amirzenoozi/persian-news-dataset


In [13]:
DATA_PATH = "/kaggle/input/datasets/amirzenoozi/persian-news-dataset/archive_v5.csv"
data = pd.read_csv(DATA_PATH, on_bad_lines="warn")
print(data.shape)
print(data.columns.tolist())
data.head(2)

(392532, 10)
['id', 'title', 'short_link', 'service', 'subgroup', 'abstract', 'body', 'tags', 'published_datetime', 'agency_name']


,id,title,short_link,service,subgroup,abstract,body,tags,published_datetime,agency_name
0,1,پیروزی قاطع مس سونگون /نارنجی‌پوشان 6 تایی کردند,http://fna.ir/f1v1mt,استانها,آذربایجان شرقی,تیم مس سونگون برابر مهمان خود به برتری پرگل دس...,به گزارش خبرگزاری فارس از تبریز، در هفته چهارم...,"لیگ برتر, مس, تبریز, اصفهان, مس سونگون, ورزقان...",2021-01-01 11:53:52,FarsNews
1,2,پخت و توزیع آش نذری به‌مناسبت سالگرد شهادت سرد...,http://fna.ir/f1v1mk,استانها,مازندران,پخت آش نذری در مرکز مازندران توسط یکی از عشاق ...,خبرگزاری فارس مازندران ـ نذری پزون یکی از سنت...,"مازندران, حاج قاسم, شهید",2021-01-01 11:51:50,FarsNews


In [ ]:
keep_cols = ["abstract", "body"]
assert all(c in data.columns for c in keep_cols), f"Missing one of {keep_cols}. Your columns: {data.columns.tolist()}"

data = data[keep_cols].copy()
data.head(10)

,abstract,body
0,تیم مس سونگون برابر مهمان خود به برتری پرگل دس...,به گزارش خبرگزاری فارس از تبریز، در هفته چهارم...
1,پخت آش نذری در مرکز مازندران توسط یکی از عشاق ...,خبرگزاری فارس مازندران ـ نذری پزون یکی از سنت...
2,رئیس جمهور آمریکا در پیامی در توییتر از تجمع ب...,به گزارش گروه بین‌الملل خبرگزاری فارس، «دونالد...
3,نماینده ولی فقیه در قرارگاه ثارالله در پی پیام...,به گزارش خبرگزاری فارس، آیت الله محمدتقی مصباح...
4,باشگاه فوتبال بایرن مونیخ با پدیده کم سن و سال...,به گزارش خبرگزاری فارس، «جمال موسیالا» در بازی...
5,آرین غلامی استاد شطرنج ایران، «انگشتری که حاج ...,به گزارش خبرنگار تعلیم و تربیت خبرگزاری فارس، ...
6,دانشگاهیان و تشکل‌های دانشگاهی دانشگاه‌های مخت...,به گزارش خبرنگار تشکل‌های دانشگاهی خبرگزاری فا...
7,دربسته خبری ۳ خبر با عناوینی همچون شمارش معکوس...,به گزارش خبرگزاری فارس از مشهد به نقل از پایگا...
8,مدیر حوزه‌های علمیه با تسلیت ارتحال آیت‌الله م...,به گزارش خبرگزاری فارس از قم، در پیام آیت الله...
9,وینگر برزیلی تیم فوتبال بارسلونا فردا تحت عمل ...,به گزارش خبرگزاری فارس، فیلیپه کوتینیو سال نو ...


In [14]:
def normalize_fa(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # Normalize Arabic characters to Persian
    text = text.replace("ي", "ی").replace("ك", "ک")
    text = text.replace("ۀ", "ه").replace("ة", "ه")

    # Remove URLs
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Keep Persian letters, digits, and basic punctuation
    text = re.sub(
        r"[^آ-ی۰-۹0-9\s\.\,\!\?\:\;\-\(\)\[\]«»\"]+",
        " ",
        text
    )

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

data["body"] = data["body"].map(normalize_fa)
data["abstract"] = data["abstract"].map(normalize_fa)

# Drop empty strings after cleaning
data = data[(data["body"].str.len() > 0) & (data["abstract"].str.len() > 0)]

# Remove duplicates
data = data.drop_duplicates(subset=["body", "abstract"]).reset_index(drop=True)

print("After cleaning:", data.shape)
data.head(2)

After cleaning: (367170, 10)


,id,title,short_link,service,subgroup,abstract,body,tags,published_datetime,agency_name
0,1,پیروزی قاطع مس سونگون /نارنجی‌پوشان 6 تایی کردند,http://fna.ir/f1v1mt,استانها,آذربایجان شرقی,تیم مس سونگون برابر مهمان خود به برتری پرگل دس...,به گزارش خبرگزاری فارس از تبریز در هفته چهارم ...,"لیگ برتر, مس, تبریز, اصفهان, مس سونگون, ورزقان...",2021-01-01 11:53:52,FarsNews
1,2,پخت و توزیع آش نذری به‌مناسبت سالگرد شهادت سرد...,http://fna.ir/f1v1mk,استانها,مازندران,پخت آش نذری در مرکز مازندران توسط یکی از عشاق ...,خبرگزاری فارس مازندران ـ نذری پزون یکی از سنت ...,"مازندران, حاج قاسم, شهید",2021-01-01 11:51:50,FarsNews


In [ ]:
USE_FRAC = 0.1
data_small = data.sample(frac=USE_FRAC, random_state=42).reset_index(drop=True)
print("Using:", data_small.shape)

Using: (36717, 10)


In [55]:
train_df, temp_df = train_test_split(data_small, test_size=0.30, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

print("train:", train_df.shape, "val:", val_df.shape, "test:", test_df.shape)

train: (25701, 10) val: (5508, 10) test: (5508, 10)


In [ ]:
train_df = train_df.copy()
val_df   = val_df.copy()
test_df  = test_df.copy()

def make_source(body: str) -> str:
    # Persian prompt
    return f"خلاصه کن: {body}"

for df in [train_df, val_df, test_df]:
    df["source_text"] = df["body"].map(make_source)
    df["target_text"] = df["abstract"]

train_df[["source_text","target_text"]].head(2)

,source_text,target_text
31759,خلاصه کن: به گزارش خبرگزاری فارس از بندرعباس م...,مدیرکل شیلات هرمزگان از برخورد قاطع و شدید با ...
6211,خلاصه کن: به گزارش خبرگزاری خبرآنلاین به نقل ا...,گروه اجرایی نمایش عروسکی «قویتر» و موسسه هنرمن...


In [57]:
train_ds = Dataset.from_pandas(train_df[["source_text","target_text"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[["source_text","target_text"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["source_text","target_text"]], preserve_index=False)

train_ds, val_ds, test_ds

(Dataset({
     features: ['source_text', 'target_text'],
     num_rows: 25701
 }),
 Dataset({
     features: ['source_text', 'target_text'],
     num_rows: 5508
 }),
 Dataset({
     features: ['source_text', 'target_text'],
     num_rows: 5508
 }))

# Fine-Tune Pars-T5

In [ ]:
!pip install -q evaluate

In [21]:
import numpy as np
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

MODEL_NAME = "Ahmad/parsT5-base"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

In [ ]:
MAX_SOURCE_LEN = 512
MAX_TARGET_LEN = 128

def preprocess_function(batch):
    model_inputs = tokenizer(
        batch["source_text"],
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=batch["target_text"],  
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
train_tok = train_ds.map(preprocess_function, batched=True, remove_columns=train_ds.column_names)
val_tok   = val_ds.map(preprocess_function, batched=True, remove_columns=val_ds.column_names)
test_tok  = test_ds.map(preprocess_function, batched=True, remove_columns=test_ds.column_names)

In [60]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

In [ ]:
!pip install hazm

In [ ]:
from hazm import word_tokenize
import numpy as np
import evaluate

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]

    # Decode predictions and labels
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    
    pred_texts = tokenizer.batch_decode(preds, skip_special_tokens=True)
    label_texts = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(
        predictions=pred_texts,
        references=label_texts,
        tokenizer=lambda text: word_tokenize(text),  # Hazm Persian tokenizer
        rouge_types=["rouge1", "rouge2", "rougeL"]
    )
    
    return {k: round(v, 4) for k, v in result.items()}


In [ ]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    pred_texts = tokenizer.batch_decode(preds, skip_special_tokens=True)
    label_texts = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    result = rouge.compute(predictions=pred_texts, references=label_texts)
    return {k: round(v, 4) for k, v in result.items()}

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="parsT5_persian_news_summarizer",
    num_train_epochs=3,
    learning_rate=1e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    generation_num_beams=4,

    logging_steps=50,
    report_to="none",
)

In [23]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
1,12.194819,5.934957,0.215300,0.075400,0.185000
2,8.168741,3.793521,0.345300,0.165200,0.305200
3,6.368636,2.873136,0.405400,0.220400,0.360200


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=9639, training_loss=10.231082750437777, metrics={'train_runtime': 17594.6332, 'train_samples_per_second': 4.382, 'train_steps_per_second': 0.548, 'total_flos': 5.279229842581094e+16, 'train_loss': 10.231082750437777, 'epoch': 3.0})

In [ ]:
test_results = trainer.evaluate(test_tok, metric_key_prefix="test")
test_results

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'test_loss': 2.8822712898254395,
 'test_rouge1': 0.4163,
 'test_rouge2': 0.2332,
 'test_rougeL': 0.3684,
 'test_runtime': 9000.2117,
 'test_samples_per_second': 0.612,
 'test_steps_per_second': 0.077}

# Inference

In [ ]:
import torch

In [ ]:
raw_text = """در سال‌های اخیر، پژوهشگران علوم اعصاب به کشفیات تازه‌ای درباره نحوه ذخیره‌سازی خاطرات در مغز دست یافته‌اند. پیش‌تر تصور می‌شد که هر خاطره در یک نقطه مشخص از مغز نگهداری می‌شود، اما یافته‌های جدید نشان می‌دهد که خاطرات در شبکه‌ای گسترده از نورون‌ها پراکنده‌اند و هنگام یادآوری، این شبکه به صورت هماهنگ فعال می‌شود. در یکی از آزمایش‌ها، دانشمندان با استفاده از تکنیک‌های تصویربرداری پیشرفته توانستند مسیر فعال‌سازی نورون‌ها را هنگام بازخوانی یک خاطره ساده مشاهده کنند و دریافتند که مغز همیشه خاطره را «همان‌طور که بوده» بازیابی نمی‌کند؛ بلکه آن را براساس تجربیات جدید بازسازی می‌کند. این پدیده سبب می‌شود که خاطرات انسانی پویا و انعطاف‌پذیر باشند، اما در عین حال، امکان خطا و تحریف نیز وجود داشته باشد. برخی متخصصان معتقدند که همین ویژگی بازسازی شوندهٔ حافظه، اساس خلاقیت انسان را تشکیل می‌دهد، زیرا مغز می‌تواند عناصر پراکنده از گذشته را ترکیب کرده و ایده‌های جدید بسازد. با این حال، هنوز پرسش‌های فراوانی بی‌پاسخ مانده است؛ از جمله اینکه دقیقاً چه عواملی مشخص می‌کنند کدام بخش‌های خاطره هنگام یادآوری برجسته‌تر می‌شوند و چرا بعضی خاطرات به مرور زمان کم‌رنگ یا کاملاً فراموش می‌شوند."""

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

cleaned_text = normalize_fa(raw_text)

inputs = tokenizer(
    cleaned_text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

inputs = {k: v.to(device) for k, v in inputs.items()}

model.eval()
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_length=128,
        min_length=30,
        num_beams=4,
        length_penalty=1.0,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(summary)

پژوهشگران علوم اعصاب به کشفیات تازه ای درباره نحوه ذخیره سازی خاطرات در مغز دست یافته اند. محققان معتقدند مغز همیشه خاطره را «همان طور که بوده» بازیابی نمی کند بلکه آن را براساس تجربیات جدید بازسازی می کند اما در عین حال امکان خطا و تحریف نیز وجود دارد.


In [27]:
import torch

raw_text = """باد عصرگاهی آرام از میان شاخه‌های بلند چنارها می‌گذشت و برگ‌های خشک را روی سنگ‌فرش باغ قدیمی می‌رقصاند. رعنا، که مدت‌ها بود پایش را به این باغ نگذاشته بود، آهسته در مسیر باریک میان درختان قدم می‌زد و با هر قدم، خاطرات سال‌های دور در ذهنش جان می‌گرفت. زمانی این‌جا پاتوق همیشگی او و دوستانش بود؛ جایی که ساعت‌ها زیر آفتاب کم‌جان پاییزی می‌نشستند و درباره رویاهای آینده حرف می‌زدند. اکنون اما سکوت سنگینی بر باغ حکم‌فرما بود، گویی همه صداهایی که زمانی در آن طنین داشتند، در غبار زمان گم شده‌اند. رعنا کنار حوض قدیمی توقف کرد. سطح آب بی‌حرکت بود و تصویر محوی از آسمان سرخ‌فام غروب در آن دیده می‌شد. او دستش را روی لبه حوض گذاشت و به انعکاس مبهم خود نگاه کرد. ناگهان به یاد قولی افتاد که سال‌ها پیش به خودش داده بود: اینکه روزی به این باغ بازگردد و تصمیم سختی را که همیشه از آن گریخته بود، بگیرد. حالا که ایستاده بود، میان غروب رو به پایان و باغی خاموش، نمی‌دانست آیا هنوز آن شجاعت را در خود دارد یا نه."""

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

cleaned_text = normalize_fa(raw_text)

inputs = tokenizer(
    cleaned_text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

inputs = {k: v.to(device) for k, v in inputs.items()}

model.eval()
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_length=128,
        min_length=30,
        num_beams=4,
        length_penalty=1.0,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(summary)

او پایش را به این باغ نگذاشته بود آهسته در مسیر باریک میان درختان قدم می زد و با هر قدم خاطرات سال های دور را در ذهنش ثبت می کرد.


In [29]:
import torch

raw_text = """در سال‌های اخیر، مطالعه بر روی «ریتم‌های درونی بدن» نشان داده است که تقریباً تمام سلول‌های انسان دارای ساعت زیستی مستقل هستند که فعالیت‌هایشان را با چرخه شب و روز هماهنگ می‌کند. این ساعت‌های سلولی، که در مجموعه‌ای از ژن‌ها و پروتئین‌های درون سلول شکل می‌گیرند، وظیفه تنظیم فرایندهایی مانند آزادسازی هورمون‌ها، دمای بدن، کیفیت خواب و حتی کارایی دستگاه ایمنی را بر عهده دارند. پژوهشگران دریافته‌اند که وقتی این ریتم‌های زیستی بر اثر بی‌خوابی، کار شیفتی یا قرار گرفتن طولانی‌مدت در نور مصنوعی مختل شود، احتمال بروز بیماری‌هایی مانند دیابت، افسردگی، چاقی و برخی اختلالات قلبی افزایش می‌یابد. یکی از نکات جالب این است که ساعت زیستی بدن تنها به نور واکنش نشان نمی‌دهد؛ بلکه عوامل دیگری مانند زمان غذا خوردن، فعالیت بدنی و حتی دمای محیط نیز بر آن اثر می‌گذارند. هم‌اکنون دانشمندان در تلاش‌اند تا با درک بهتر این سازوکارها، روش‌هایی برای تنظیم دوباره ساعت‌های درونی بدن طراحی کنند تا بتوانند درمان‌های مؤثرتری برای اختلالات خواب و بیماری‌های وابسته به ریتم شبانه‌روزی ارائه دهند."""

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

cleaned_text = normalize_fa(raw_text)

inputs = tokenizer(
    cleaned_text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

inputs = {k: v.to(device) for k, v in inputs.items()}

model.eval()
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=60,
        min_new_tokens=15,
        num_beams=5,
        length_penalty=2.0,
        no_repeat_ngram_size=3,
        repetition_penalty=1.2,
        early_stopping=True
    )

summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(summary)

محققان دریافته اند که ساعت زیستی بدن تنها به نور واکنش نشان نمی دهد بلکه عوامل دیگری مانند زمان غذا خوردن فعالیت بدنی و حتی دمای محیط نیز بر آن اثر می گذارند.
